# NES-VMC（Natural Excited State Variational Monte Carlo）算法实现

本 notebook 实现基于原生 JAX 和部分 NetKet 的 NES-VMC 算法，用于计算量子多体系统（如 H₂ 分子）的激发态能量。

## 目录

1. [环境配置与 FCI 基准](#1-环境配置)
2. [神经网络 Ansatz 定义](#2-神经网络-Ansatz)
3. [NES 总 Ansatz](#3-NES-总-Ansatz)
4. [哈密顿量作用计算](#4-哈密顿量作用计算)
5. [局域能量矩阵](#5-局域能量矩阵)
6. [损失函数与梯度](#6-损失函数与梯度)
7. [训练循环](#7-训练循环)
8. [结果可视化](#8-结果可视化)
9. [代码解读](#9-代码解读)

## 1. 环境配置与 FCI 基准

首先导入必要的库并设置 H₂ 分子系统。

In [1]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.5.3
NetKet version: 3.18


In [2]:
# ===================== H₂ 分子定义 & FCI 基准 =====================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV")

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


In [3]:
# ===================== NetKet 哈密顿量和采样器 =====================
ha = nkx.operator.from_pyscf_molecule(mol)

hi = nkx.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

print(f"Hilbert space size: {hi.size}")

Hilbert space size: 4


/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_84777/2366098020.py:4: DeprecationWarning: netket.experimental.hilbert.SpinOrbitalFermions is deprecated: use netket.hilbert.SpinOrbitalFermions (netket >= 3.12)
  hi = nkx.hilbert.SpinOrbitalFermions(


## 2. 神经网络 Ansatz 定义

定义单态 Ansatz，用于近似单个量子态的波函数。

In [4]:
# ===================== 单态 Ansatz =====================
class SingleStateAnsatz(nnx.Module):
    """单态 Ansatz：适配费米子系统的复数值 FFNN"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_spin_orbitals = n_spin_orbitals
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array) -> jax.Array:
        """输入：x = 单组自旋轨道占据态（形状 [n_spin_orbitals,]）
        输出：复数值波函数值 ψ(x)"""
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

print("SingleStateAnsatz 定义完成!")

SingleStateAnsatz 定义完成!


## 3. NES 总 Ansatz

定义 NES-VMC 的核心：**K 个单态 Ansatz 的行列式**。

数学形式：
$$
\Psi(\mathbf{x}) = \det\begin{pmatrix}
\psi_1(x^1) & \psi_2(x^1) \\
\psi_1(x^2) & \psi_2(x^2)
\end{pmatrix}
$$

In [5]:
# ===================== NES 总 Ansatz =====================
class NESTotalAnsatz(nnx.Module):
    """NES-VMC 总 Ansatz：K 个单态 Ansatz 的行列式"""
    def __init__(self, n_spin_orbitals: int, n_states: int = 2, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals

        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)

        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(keys[i]))
            for i in range(n_states)
        ]

    def __call__(self, x_batch: jax.Array) -> tuple[jax.Array, jax.Array]:
        """x_batch: (n_states, n_spin_orbitals)
        
        输出：
            psi_total : 行列式标量（给 NetKet 采样）
            M_matrix  : K×K 矩阵（给局域能量矩阵计算）
        """
        if x_batch.shape[0] != self.n_states:
            raise ValueError(f"x_batch.shape[0] != {self.n_states}")

        K = self.n_states
        M = []
        for i in range(K):
            for j in range(K):
                psi_i_xj = self.single_ansatz_list[j](x_batch[i])
                M.append(psi_i_xj)

        M = jnp.stack(M, axis=0)
        M_matrix = M.reshape(K, K)
        psi_total = jnp.linalg.det(M_matrix)
        return psi_total, M_matrix

print("NESTotalAnsatz 定义完成!")

NESTotalAnsatz 定义完成!


## 4. 哈密顿量作用计算

实现 $\hat{H}\psi(x)$ 和 $\hat{H}\Psi(\mathbf{x})$ 的计算。

In [6]:
# ===================== 哈密顿量作用计算 =====================
def Ham_psi(ha, model, x):
    """计算 Hψ(x)"""
    x_primes, mels = ha.get_conn(x)
    psi_values = jax.vmap(model)(x_primes)
    H_psi_x = jnp.sum(mels * psi_values)
    return H_psi_x


def Ham_Psi(ha, total_ansatz, x):
    """计算扩展哈密顿量作用在总 Ansatz 上的矩阵"""
    k = total_ansatz.n_states
    if x.shape[0] != k:
        raise ValueError(f"Input array must have shape ({k},) but got shape {x.shape}")

    H_psi_x_i = []
    for i in range(k):
        tmp = []
        for j in range(k):
            ele = Ham_psi(ha, model=total_ansatz.single_ansatz_list[j], x=x[i])
            tmp.append(ele)
        H_psi_x_i.append(tmp)

    HPsi = jnp.array(H_psi_x_i).reshape(k, k)
    return HPsi

print("哈密顿量作用计算函数定义完成!")

哈密顿量作用计算函数定义完成!


## 5. 局域能量矩阵

计算局域能量矩阵：

$$E_L(\mathbf{x}) = \Psi^{-1}(\mathbf{x})\tilde{H}\Psi(\mathbf{x})$$

这是 NES-VMC 的核心量。

In [7]:
# ===================== 局域能量矩阵计算 =====================
def compute_local_energy_matrix(ha, total_ansatz, x_batch):
    """
    计算局域能量矩阵 E_L(x) = Ψ^{-1}(x) HΨ(x)
    
    参数：
        ha: 哈密顿量
        total_ansatz: NES 总 Ansatz
        x_batch: 批次样本，形状 (n_states, n_spin_orbitals)
    
    返回：
        E_L: K×K 局域能量矩阵
        psi_total: 行列式值
    """
    psi_total, M_matrix = total_ansatz(x_batch)
    H_Psi = Ham_Psi(ha, total_ansatz, x_batch)

    E_L = jnp.linalg.solve(M_matrix, H_Psi.T).T
    return E_L, psi_total

print("局域能量矩阵计算函数定义完成!")

局域能量矩阵计算函数定义完成!


## 6. 损失函数与梯度

NES-VMC 的损失函数为局域能量矩阵的迹：

$$\mathcal{L} = \mathrm{Tr}(E_L)$$

梯度公式：

$$\nabla_\theta \mathcal{L} = 2\langle (E_L - \bar{E}_L) \nabla_\theta \log|\Psi| \rangle$$

In [8]:
# ===================== NES 损失函数与梯度计算 =====================

def compute_local_energy_matrix_from_params_safe(ha, graphdef, params, x_batch, n_states, eps=1e-8):
    """从参数计算局域能量矩阵，带数值稳定性保护"""
    merged_model = nnx.merge(graphdef, params)

    K = n_states
    M = []
    for i in range(K):
        for j in range(K):
            psi_i_xj = merged_model.single_ansatz_list[j](x_batch[i])
            M.append(psi_i_xj)

    M_matrix = jnp.stack(M).reshape(K, K)

    det_M = jnp.linalg.det(M_matrix)
    abs_det = jnp.abs(det_M)

    M_reg = M_matrix
    if abs_det < eps:
        M_reg = M_matrix + eps * jnp.eye(K)

    H_Psi = []
    for i in range(K):
        row = []
        for j in range(K):
            x_primes, mels = ha.get_conn(x_batch[i])
            psi_values = jax.vmap(lambda x: merged_model.single_ansatz_list[j](x))(x_primes)
            H_psi = jnp.sum(mels * psi_values)
            row.append(H_psi)
        H_Psi.append(row)

    H_Psi_matrix = jnp.array(H_Psi).reshape(K, K)

    E_L = jnp.linalg.solve(M_reg, H_Psi_matrix.T).T
    psi_total = jnp.linalg.det(M_matrix)

    return E_L, psi_total, det_M


def sample_nes_batches(sampler, machine_fn, params, sampler_state, n_samples, K, n_spin_orbitals):
    """从扩展希尔伯特空间采样 NES 批次"""
    samples_list = []

    for _ in range(n_samples):
        sample, sampler_state = sampler.sample(machine_fn, params, state=sampler_state)
        samples_list.append(sample.reshape(-1, n_spin_orbitals))

    all_samples = jnp.stack(samples_list).reshape(-1, n_spin_orbitals)

    x_batches = []
    for i in range(n_samples // K):
        indices = jnp.arange(i * K, (i + 1) * K)
        batch = all_samples[indices]
        x_batches.append(batch)

    return jnp.stack(x_batches), sampler_state


def compute_nes_loss_and_grad(ha, graphdef, params, sigma_batches, n_states):
    """计算 NES-VMC 的损失函数和梯度"""
    E_L_list = []

    for i in range(sigma_batches.shape[0]):
        E_L, _ = compute_local_energy_matrix_from_params(
            ha, graphdef, params, sigma_batches[i], n_states
        )
        E_L_list.append(E_L)

    E_L_batch = jnp.stack(E_L_list)
    loss = jnp.trace(jnp.mean(E_L_batch, axis=0))

    grads = []
    for i in range(sigma_batches.shape[0]):
        def loss_per_sample(p):
            E_L_s, _ = compute_local_energy_matrix_from_params(
                ha, graphdef, p, sigma_batches[i], n_states
            )
            return jnp.trace(E_L_s)

        grad_i = jax.grad(loss_per_sample, holomorphic=True)(params)
        grads.append(grad_i)

    grad = jax.tree.map(
        lambda *x: jnp.mean(jnp.stack(x), axis=0),
        *grads
    )

    return loss, grad, E_L_batch


def extract_eigenvalues(E_L_matrices):
    """从局域能量矩阵提取本征值（激发态能量）"""
    E_L_mean = jnp.mean(E_L_matrices, axis=0)
    eigenvalues, eigenvectors = jnp.linalg.eigh(E_L_mean)
    return eigenvalues, eigenvectors, E_L_mean

print("损失函数与梯度计算函数定义完成!")

损失函数与梯度计算函数定义完成!


## 7. 训练循环

完整的 NES-VMC 训练循环。

In [9]:
# ===================== 训练参数设置 =====================
K = 2  # 态数量
n_spin_orbitals = 4
hidden_dim = 16
learning_rate = 0.01
n_iterations = 100
n_samples = 1008

print(f"训练参数:")
print(f"  - K (态数量): {K}")
print(f"  - n_spin_orbitals: {n_spin_orbitals}")
print(f"  - hidden_dim: {hidden_dim}")
print(f"  - 学习率: {learning_rate}")
print(f"  - 迭代次数: {n_iterations}")
print(f"  - 样本数: {n_samples}")

训练参数:
  - K (态数量): 2
  - n_spin_orbitals: 4
  - hidden_dim: 16
  - 学习率: 0.01
  - 迭代次数: 100
  - 样本数: 1008


In [10]:
# ===================== 初始化模型和优化器 =====================
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=n_spin_orbitals,
    n_states=K,
    hidden_dim=hidden_dim,
    rngs=nnx.Rngs(42)
)

graphdef, params = nnx.split(total_ansatz)

# 创建用于采样的 machine 函数（单态）
single_machine = total_ansatz.single_ansatz_list[0]
single_graphdef, single_state = nnx.split(single_machine)

@jax.jit
def sampling_machine(params, sigma):
    model = nnx.merge(single_graphdef, params)
    return model(sigma)

sampler_state = sampler.init_state(sampling_machine, single_state, seed=42)

optimizer = optax.sgd(learning_rate=learning_rate)
opt_state = optimizer.init(params)

print("模型和优化器初始化完成!")

模型和优化器初始化完成!


In [11]:
# ===================== 训练循环 =====================
print("\n" + "="*60)
print("开始 NES-VMC 训练")
print("="*60)

history = {
    'step': [],
    'loss': [],
    'eigenvalues': [],
    'error_0': [],
    'error_1': []
}

for step in tqdm(range(n_iterations)):
    # 采样
    sigma_batches, sampler_state = sample_nes_batches(
        sampler, sampling_machine, single_state, sampler_state,
        n_samples, K, n_spin_orbitals
    )

    # 计算损失函数和梯度
    loss, grad, E_L_batch = compute_nes_loss_and_grad(
        ha, graphdef, params, sigma_batches, K
    )

    # 提取本征值
    eigenvalues, eigenvectors, E_L_mean = extract_eigenvalues(E_L_batch)

    # 参数更新
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    if step % 10 == 0:
        history['step'].append(step)
        history['loss'].append(float(loss.real))
        history['eigenvalues'].append([float(e) for e in eigenvalues])
        history['error_0'].append(float(abs(eigenvalues[0] - E_fcis[0])))
        history['error_1'].append(float(abs(eigenvalues[1] - E_fcis[1])))

        print(f"\nStep {step:3d} | Loss: {loss.real:.8f}")
        print(f"  本征值: E_0 = {eigenvalues[0]:.8f} (FCI: {E_fcis[0]:.8f}, Error: {abs(eigenvalues[0] - E_fcis[0]):.6f})")
        print(f"          E_1 = {eigenvalues[1]:.8f} (FCI: {E_fcis[1]:.8f}, Error: {abs(eigenvalues[1] - E_fcis[1]):.6f})")

print("\n" + "="*60)
print("训练完成!")
print("="*60)


开始 NES-VMC 训练


  0%|          | 0/100 [00:01<?, ?it/s]


NameError: name 'compute_local_energy_matrix_from_params' is not defined

## 8. 结果可视化

In [ ]:
# ===================== 结果可视化 =====================
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history['step'], history['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Iteration Step')
axes[0, 0].set_ylabel('Loss (Tr(E_L))')
axes[0, 0].set_title('NES-VMC Loss Convergence')
axes[0, 0].grid(True, alpha=0.3)

eigenvalues_array = jnp.array(history['eigenvalues'])
for i in range(K):
    axes[0, 1].plot(history['step'], eigenvalues_array[:, i],
                    label=f'E_{i}', linewidth=2)
axes[0, 1].axhline(y=E_fcis[0], color='r', linestyle='--',
                   label=f'FCI E0: {E_fcis[0]:.4f}')
axes[0, 1].axhline(y=E_fcis[1], color='g', linestyle='--',
                   label=f'FCI E1: {E_fcis[1]:.4f}')
axes[0, 1].set_xlabel('Iteration Step')
axes[0, 1].set_ylabel('Energy (Ha)')
axes[0, 1].set_title('Eigenvalue Convergence')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].semilogy(history['step'], history['error_0'], 'b-',
                     linewidth=2, label='Error E_0')
axes[1, 0].semilogy(history['step'], history['error_1'], 'r-',
                     linewidth=2, label='Error E_1')
axes[1, 0].set_xlabel('Iteration Step')
axes[1, 0].set_ylabel('Absolute Error (Ha)')
axes[1, 0].set_title('Error vs FCI Reference')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, which='both')

axes[1, 1].bar(['FCI E0', 'NES E0', 'FCI E1', 'NES E1'],
               [E_fcis[0], eigenvalues_array[-1, 0],
                E_fcis[1], eigenvalues_array[-1, 1]],
               color=['red', 'blue', 'red', 'blue'],
               alpha=0.7)
axes[1, 1].set_ylabel('Energy (Ha)')
axes[1, 1].set_title('Final Energy Comparison')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('nes_vmc_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n结果图表已保存为 'nes_vmc_results.png'")

## 9. 代码解读

### 9.1 NES-VMC 核心思想

NES-VMC 将原系统前 $K$ 个激发态的求解问题等价转化为一个"扩展系统"的基态求解问题。

### 9.2 TotalAnsatz 的构成

总 Ansatz 定义为矩阵 $\Psi(\mathbf{x}) \in \mathbb{C}^{K \times K}$ 的行列式：

$$
\Psi(\mathbf{x}) = \det\begin{pmatrix}
\psi_1(x^1) & \psi_2(x^1) \\
\psi_1(x^2) & \psi_2(x^2)
\end{pmatrix}
$$

### 9.3 损失函数

NES-VMC 的目标函数为扩展哈密顿量关于总 Ansatz 的 Rayleigh 商：

$$
\mathcal{L} = \frac{\langle\Psi|\tilde{H}|\Psi\rangle}{\langle\Psi|\Psi\rangle} = \mathrm{Tr}(S^{-1}\hat{H})
$$

### 9.4 激发态能量提取

训练完成后，通过对局域能量矩阵对角化获得各激发态能量：

$$
\bar{E}_L = U\Lambda U^{-1}
$$

其中 $\Lambda = \mathrm{diag}(E_1, E_2, \dots, E_K)$ 包含按能量排序的本征值。

## 参考资料

- [NES-VMC 原始论文]
- [VMC 梯度计算](../0504/VMC 的梯度计算.md)
- [纯 JAX 实现 VMC](../0508/纯 JAX 实现 VMC 梯度下降.md)
- [NetKet 官方文档](https://www.netket.org/)